<a href="https://colab.research.google.com/github/ShashankS1ngh/GSDT/blob/main/Fake_Image_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch pillow accelerate


In [3]:
from google.colab import files
from transformers import pipeline

# 1. Upload an image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]  # get the uploaded filename

# 2. Load the detector
pipe = pipeline("image-classification", model="Ateeqq/ai-vs-human-image-detector")

# 3. Run detection
result = pipe(image_path)
print("📊 Raw Result:", result)


Saving Beach.jpg to Beach.jpg


Device set to use cpu


📊 Raw Result: [{'label': 'hum', 'score': 0.999430239200592}, {'label': 'ai', 'score': 0.0005697289598174393}]


In [5]:
from google.colab import files
from transformers import pipeline

# 1. Upload an image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]  # get the uploaded filename

# 2. Load the detector
pipe = pipeline("image-classification", model="Ateeqq/ai-vs-human-image-detector")

# 3. Run detection
result = pipe(image_path)
print("📊 Raw Result:", result)


Saving Gemini_Generated_Image_841vlb841vlb841v.png to Gemini_Generated_Image_841vlb841vlb841v.png


Device set to use cpu


📊 Raw Result: [{'label': 'ai', 'score': 0.8161134123802185}, {'label': 'hum', 'score': 0.1838865727186203}]


In [12]:
import gradio as gr
from transformers import pipeline

MODELS = [
    "Ateeqq/ai-vs-human-image-detector",
    "umm-maybe/AI-image-detector",
]

pipes = {}
for name in MODELS:
    try:
        pipes[name] = pipeline("image-classification", model=name)
    except Exception as e:
        print(f"⚠️ Could not load {name}: {e}")

def detect_all(image):
    results = []
    for model_name, pipe in pipes.items():
        try:
            out = pipe(image)
            scores = {r["label"]: float(r["score"]) for r in out}
            results.append(scores)
        except Exception as e:
            results.append({"error": 1.0})  # so gr.Label shows something
    return results

demo = gr.Interface(
    fn=detect_all,
    inputs=gr.Image(type="filepath"),
    outputs=[gr.Label(num_top_classes=2, label=model) for model in MODELS],
    title="AI vs Human Image Detector (Multi-model)",
    description="Upload an image. Each model will output its probabilities visually."
)

if __name__ == "__main__":
    demo.launch(debug=True)


Device set to use cpu
Device set to use cpu


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f3658170b51374d04a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f3658170b51374d04a.gradio.live


In [13]:
# save as multi_model_ai_detector.py and run
import gradio as gr
from transformers import pipeline
from PIL import Image, ExifTags
import numpy as np
from collections import Counter

# --- Curated models to try (public)
CANDIDATE_MODELS = [
    "Ateeqq/ai-vs-human-image-detector",          # community model (may be overconfident)
    "jacoballessio/ai-image-detect-distilled",    # distilled ensemble detector (often better)
    "prithivMLmods/Deep-Fake-Detector-Model",     # another community detector
    # add others you find / have access to
]

# --- Load pipelines (CPU). If a model fails to load, skip it but do not crash.
pipes = {}
for m in CANDIDATE_MODELS:
    try:
        print(f"Loading {m} ...")
        pipes[m] = pipeline("image-classification", model=m, device=-1)  # CPU
    except Exception as e:
        print(f"⚠️ Could not load {m}: {e}")

if len(pipes) == 0:
    raise SystemExit("No models loaded. Check network/auth or add local models.")

loaded_models = list(pipes.keys())

# --- Helpers: map varied labels -> canonical buckets AI / Human
AI_KEYWORDS = ["ai", "artificial", "synthetic", "generated", "fake", "gan", "diffusion"]
HUM_KEYWORDS = ["human", "hum", "real", "photo", "photograph"]

def canonicalize_scores(raw_scores: dict):
    """
    raw_scores: {'ai':0.98, 'hum':0.02} OR other label names
    returns: {'AI': float, 'Human': float} (normalized)
    """
    ai = 0.0
    hum = 0.0
    for label, score in raw_scores.items():
        l = label.lower()
        if any(k in l for k in AI_KEYWORDS):
            ai += score
        if any(k in l for k in HUM_KEYWORDS):
            hum += score
    # fallback: if none matched, use top label mapping heuristics
    if ai == 0 and hum == 0:
        top = max(raw_scores, key=raw_scores.get)
        t = top.lower()
        if any(k in t for k in AI_KEYWORDS):
            ai = raw_scores[top]
        else:
            hum = raw_scores[top]
    total = ai + hum
    if total == 0:
        return {"AI": 0.5, "Human": 0.5}
    return {"AI": ai / total, "Human": hum / total}

def read_exif_text(pil_img):
    try:
        exif = pil_img._getexif()
        if not exif:
            return "No EXIF metadata found."
        out = []
        for k, v in exif.items():
            tag = ExifTags.TAGS.get(k, k)
            out.append(f"{tag}: {v}")
        return "\n".join(out)
    except Exception:
        return "Could not read EXIF."

# --- Main detection function: returns results in same order as gradio outputs
def detect_all(filepath):
    pil = Image.open(filepath).convert("RGB")
    per_model_canonical = []
    top_votes = []

    for model_name, pipe in pipes.items():
        try:
            out = pipe(pil)
            # out is list of dicts: [{'label': 'ai', 'score': 0.99}, ...]
            raw = {d["label"]: float(d["score"]) for d in out}
            canon = canonicalize_scores(raw)
            # round for better visual
            per_model_canonical.append({"AI": round(canon["AI"], 4), "Human": round(canon["Human"], 4)})
            top_votes.append("AI" if canon["AI"] >= canon["Human"] else "Human")
        except Exception as e:
            # keep UI stable: show an 'error' bar
            per_model_canonical.append({"error": 1.0})
            top_votes.append("Unknown")

    # Consensus (average of canonical scores)
    avg_ai = float(np.mean([c["AI"] for c in per_model_canonical if "AI" in c]))
    avg_h = float(np.mean([c["Human"] for c in per_model_canonical if "Human" in c]))
    consensus = {"AI": round(avg_ai, 4), "Human": round(avg_h, 4)}

    # Majority vote
    votes = [v for v in top_votes if v in ("AI", "Human")]
    if votes:
        most = Counter(votes).most_common(1)[0][0]
    else:
        most = "Uncertain"

    # Final verdict logic (simple, conservative)
    if abs(consensus["AI"] - consensus["Human"]) < 0.15:
        final_verdict = f"Uncertain — models disagree (AI {consensus['AI']:.2%} vs Human {consensus['Human']:.2%})"
    else:
        final_verdict = f"{most} (consensus: AI {consensus['AI']:.2%}, Human {consensus['Human']:.2%})"

    exif_text = read_exif_text(pil)
    # return: per-model dicts in same order, then consensus dict, then final verdict text, then exif text
    return per_model_canonical + [consensus] + [final_verdict] + [exif_text]

# --- Build Gradio outputs dynamically to match successfully loaded models
label_outputs = [gr.Label(num_top_classes=2, label=model) for model in loaded_models]
label_outputs.append(gr.Label(num_top_classes=2, label="Consensus (avg)"))
label_outputs.append(gr.Textbox(label="Final verdict (majority + confidence)"))
label_outputs.append(gr.Textbox(label="EXIF / metadata (if any)"))

demo = gr.Interface(
    fn=detect_all,
    inputs=gr.Image(type="filepath"),
    outputs=label_outputs,
    title="Multi-model AI vs Human image detector (visual per-model bars + consensus)",
    description=(
        "Runs multiple detectors and shows a separate probability bar for each model, "
        "then an averaged consensus. Models are community models and can still be wrong. "
        "If consensus is 'Uncertain', treat result as unreliable and inspect manually."
    ),
    allow_flagging="never",
)

if __name__ == '__main__':
    demo.launch(debug=True)


Loading Ateeqq/ai-vs-human-image-detector ...


Device set to use cpu


Loading jacoballessio/ai-image-detect-distilled ...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/58.3M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:414: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e4128044b81f7e181d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e4128044b81f7e181d.gradio.live
